# 05 · Predicted Coding Questions: Scotiabank EES Onsite
**Monday Oct 26, 11:05.** Expect ~20–30 min of coding, Easy–Medium, possibly **on paper or a whiteboard**.

These are predictions, not leaked questions. They're chosen because trading desks like problems about **price series, streams and small designs**, and because your Squarepoint round was a design question (a transactional key-value store).

**Format (same as notebook 02):** Question → your attempt + tests → 💡 Hint → ✅ Answer → 🔑 Takeaway. Run the setup cell first.

### If you only have time for five
**A1 → A2, B1, A3, C1, C2.** Together they cover the patterns behind almost everything here:

| Pattern | Problems |
|---|---|
| Running min / max | A1, A2 |
| Monotonic stack | A3, A4 |
| Heap (+ lazy deletion) | A5, B3, B4, C5, D3 |
| Deque window | B1, B2, C4 |
| Hash map (+ binary search / order) | C1, C2, C3, D1 |
| Sort + sweep | D2 |

### Sunday schedule
| Time | Problems |
|---|---|
| Morning (2.5 h) | A1, A2, B1, A3, A4 |
| Early afternoon (2.5 h) | C1, C2, B3, C3, C4 |
| Late afternoon (2 h) | **C5 on paper**, A5, B2 |
| Evening (1 h) | Redo anything you starred, **out loud**, 25-min timer |

### Say this every time (on paper too)
1. **Repeat the problem** and ask 1–2 clarifying questions (empty input? duplicates? ties? sorted?).
2. **Brute force** + its complexity, in one sentence.
3. **Better idea** + complexity, *before* writing code.
4. Write it with clear names.
5. **Trace a small example by hand** and fix bugs out loud.

## Setup: run this first

In [ ]:
import copy, heapq, bisect
from collections import deque, defaultdict, OrderedDict

def _approx(x):
    if isinstance(x, float): return round(x, 5)
    if isinstance(x, (list, tuple)): return type(x)(_approx(v) for v in x)
    if isinstance(x, dict): return {k: _approx(v) for k, v in x.items()}
    return x

def check(fn, cases):
    """cases = [(args_tuple, expected), ...]"""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ case {i}: raised {type(e).__name__}: {e}"); continue
        if _approx(got) == _approx(expected): passed += 1
        else: print(f"  ✗ case {i}: args={args!r}\n      expected={expected!r}\n      got     ={got!r}")
    print(f"{'✅' if passed == len(cases) else '❌'} {passed}/{len(cases)} passed")
    return passed == len(cases)

def run_ops(cls, ops, args):
    """LeetCode-style design tester: ops[0] is the constructor."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

def check_ops(cls_name, ops, args, expected):
    cls = globals()[cls_name]
    return check(lambda o, a: run_ops(cls, o, a), [((ops, args), expected)])

---
# A · Price-series problems (most likely)

## A1 · Best Time to Buy and Sell Stock
`LC 121 · Easy`

**Why it might come up:** The classic trading-desk opener. Expect the follow-up (A2) right after.

**Problem.** `prices[i]` is the price on day i. Buy once, sell once later. Return the max profit (0 if no profit is possible).

**Example.** `[7,1,5,3,6,4]` → `5` (buy at 1, sell at 6) · `[7,6,4,3,1]` → `0`

In [ ]:
def maxProfit(prices: list[int]) -> int:
    pass

# --- tests ---
check(maxProfit, [(([7,1,5,3,6,4],), 5), (([7,6,4,3,1],), 0), (([2,4,1],), 2), (([5],), 0)])

<details><summary>💡 <b>Hint</b></summary>

Walk once. Keep the **lowest price so far**. At each day, the best sale today is `price - lowest_so_far`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxProfit(prices: list[int]) -> int:
    lowest, best = float("inf"), 0
    for p in prices:
        lowest = min(lowest, p)
        best = max(best, p - lowest)
    return best
```

**Complexity:** O(n) time, O(1) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Brute force is O(n²) (try every buy/sell pair). Say that first, then: "I only need the cheapest price before today", which is the running-minimum pattern.
</details>

## A2 · Best Time to Buy and Sell Stock II (unlimited trades)
`LC 122 · Medium`

**Why it might come up:** The usual follow-up to A1. Tests whether you can see a greedy shortcut.

**Problem.** Same prices, but you may buy and sell as many times as you like (hold at most one share at a time). Max profit?

**Example.** `[7,1,5,3,6,4]` → `7` (1→5 and 3→6) · `[1,2,3,4,5]` → `4`

In [ ]:
def maxProfit2(prices: list[int]) -> int:
    pass

# --- tests ---
check(maxProfit2, [(([7,1,5,3,6,4],), 7), (([1,2,3,4,5],), 4), (([7,6,4,3,1],), 0)])

<details><summary>💡 <b>Hint</b></summary>

Any rise from one day to the next can be captured. Add up every positive `prices[i] - prices[i-1]`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxProfit2(prices: list[int]) -> int:
    return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, len(prices)))
```

**Complexity:** O(n), O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Greedy: a long climb 1→5 earns the same as 1→2→3→4→5 step by step. If they add a fee or a cooldown, it becomes DP (notebook 02, D5).
</details>

## A3 · Online Stock Span
`LC 901 · Medium`

**Why it might come up:** A streaming price question with a monotonic stack, a pattern interviewers love.

**Problem.** Prices arrive one per day. `next(price)` returns the **span**: how many consecutive days up to and including today had a price ≤ today's price.

**Example.** Prices 100, 80, 60, 70, 60, 75, 85 → spans 1, 1, 1, 2, 1, 4, 6

In [ ]:
class StockSpanner:
    def __init__(self):
        pass
    def next(self, price: int) -> int:
        pass

# --- tests ---
check_ops("StockSpanner", ["StockSpanner","next","next","next","next","next","next","next"], [[],[100],[80],[60],[70],[60],[75],[85]], [None,1,1,1,2,1,4,6])

<details><summary>💡 <b>Hint</b></summary>

Keep a stack of `(price, span)` with **decreasing** prices. For a new price, pop every entry with price ≤ it and add their spans to yours.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class StockSpanner:
    def __init__(self):
        self.stack = []                      # (price, span), prices strictly decreasing

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]      # absorb smaller days
        self.stack.append((price, span))
        return span
```

**Complexity:** O(1) amortized per call (each price is pushed and popped once), O(n) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**Monotonic stack**: whenever you need "the previous bigger/smaller element", keep a stack that stays sorted and pop what can never matter again.
</details>

## A4 · Daily Temperatures
`LC 739 · Medium`

**Why it might come up:** The same monotonic-stack pattern. Rephrased for trading: "how many days until a higher price?"

**Problem.** For each day, return how many days you wait for a warmer (higher) value, or 0 if none.

**Example.** `[73,74,75,71,69,72,76,73]` → `[1,1,4,2,1,1,0,0]`

In [ ]:
def dailyTemperatures(t: list[int]) -> list[int]:
    pass

# --- tests ---
check(dailyTemperatures, [(([73,74,75,71,69,72,76,73],), [1,1,4,2,1,1,0,0]), (([30,40,50,60],), [1,1,1,0]), (([30,20,10],), [0,0,0])])

<details><summary>💡 <b>Hint</b></summary>

Stack of **indices** still waiting for a higher value. When today is higher than the top, pop it and record `today - index`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def dailyTemperatures(t: list[int]) -> list[int]:
    ans = [0] * len(t)
    stack = []                              # indices with decreasing values
    for i, x in enumerate(t):
        while stack and t[stack[-1]] < x:
            j = stack.pop()
            ans[j] = i - j
        stack.append(i)
    return ans
```

**Complexity:** O(n), O(n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Store **indices**, not values, when the answer is a distance. Same pattern as A3.
</details>

## A5 · Stock Price Fluctuation
`LC 2034 · Medium`

**Why it might come up:** A realistic design question: price updates can arrive late or be corrected. Uses a hashmap + two heaps with **lazy deletion**.

**Problem.** `update(timestamp, price)` sets (or **corrects**) the price at a timestamp. `current()` = price at the latest timestamp. `maximum()` / `minimum()` = max/min over the current (corrected) prices.

**Example.** update(1,10), update(2,5), current()→5, maximum()→10, update(1,3) (correction!), maximum()→5, update(4,2), minimum()→2

In [ ]:
class StockPrice:
    def __init__(self):
        pass
    def update(self, timestamp: int, price: int) -> None:
        pass
    def current(self) -> int:
        pass
    def maximum(self) -> int:
        pass
    def minimum(self) -> int:
        pass

# --- tests ---
check_ops("StockPrice", ["StockPrice","update","update","current","maximum","update","maximum","update","minimum"], [[],[1,10],[2,5],[],[],[1,3],[],[4,2],[]], [None,None,None,5,10,None,5,None,2])

<details><summary>💡 <b>Hint</b></summary>

`prices = {timestamp: price}`, `latest` timestamp. Push `(price, ts)` into a min-heap and `(-price, ts)` into a max-heap on every update. When reading the top, **pop it if it's stale** (`prices[ts] != price`).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class StockPrice:
    def __init__(self):
        self.prices = {}                    # timestamp -> current price
        self.latest = 0
        self.min_h, self.max_h = [], []     # (price, ts) and (-price, ts)

    def update(self, timestamp: int, price: int) -> None:
        self.prices[timestamp] = price
        self.latest = max(self.latest, timestamp)
        heapq.heappush(self.min_h, (price, timestamp))
        heapq.heappush(self.max_h, (-price, timestamp))

    def current(self) -> int:
        return self.prices[self.latest]

    def maximum(self) -> int:
        while -self.max_h[0][0] != self.prices[self.max_h[0][1]]:   # stale entry
            heapq.heappop(self.max_h)
        return -self.max_h[0][0]

    def minimum(self) -> int:
        while self.min_h[0][0] != self.prices[self.min_h[0][1]]:
            heapq.heappop(self.min_h)
        return self.min_h[0][0]
```

**Complexity:** update O(log n); max/min amortized O(log n); current O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**Lazy deletion**: heaps can't remove an arbitrary item cheaply, so leave old entries and discard them when they reach the top. The same trick works in an order book for cancelled orders.
</details>

---
# B · Streams and windows

## B1 · Moving Average from Data Stream
`LC 346 · Easy`

**Why it might come up:** "Compute the 20-tick moving average of a price stream." Very natural for a trading desk.

**Problem.** `MovingAverage(size)`; `next(val)` returns the average of the last `size` values (or of all values if fewer).

**Example.** size 3: next(1)→1.0, next(10)→5.5, next(3)→4.667, next(5)→6.0

In [ ]:
class MovingAverage:
    def __init__(self, size: int):
        pass
    def next(self, val: int) -> float:
        pass

# --- tests ---
check_ops("MovingAverage", ["MovingAverage","next","next","next","next"], [[3],[1],[10],[3],[5]], [None,1.0,5.5,14/3,6.0])

<details><summary>💡 <b>Hint</b></summary>

A `deque` of the window plus a **running sum**: add the new value, subtract the value that falls out.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class MovingAverage:
    def __init__(self, size: int):
        self.size, self.q, self.total = size, deque(), 0

    def next(self, val: int) -> float:
        self.q.append(val)
        self.total += val
        if len(self.q) > self.size:
            self.total -= self.q.popleft()
        return self.total / len(self.q)
```

**Complexity:** O(1) per call, O(size) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Never re-sum the window (that's O(k) per tick). Running sum + deque = O(1). Follow-ups: VWAP over a window (keep Σprice·qty and Σqty), or a time-based window (pop while `oldest_ts <= now - T`).
</details>

## B2 · Sliding Window Maximum
`LC 239 · Hard (but common)`

**Why it might come up:** "Highest price in every 5-minute window." The monotonic-deque trick is worth knowing.

**Problem.** Return the max of every window of size k.

**Example.** `[1,3,-1,-3,5,3,6,7], k=3` → `[3,3,5,5,6,7]`

In [ ]:
def maxSlidingWindow(nums: list[int], k: int) -> list[int]:
    pass

# --- tests ---
check(maxSlidingWindow, [(([1,3,-1,-3,5,3,6,7], 3), [3,3,5,5,6,7]), (([1], 1), [1]), (([9,8,7,6], 2), [9,8,7])])

<details><summary>💡 <b>Hint</b></summary>

Deque of **indices** whose values are decreasing. Pop from the back anything ≤ the new value (it can never be a max again). Pop from the front if it left the window. The front is the max.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxSlidingWindow(nums: list[int], k: int) -> list[int]:
    dq, out = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                  # front fell out of the window
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out
```

**Complexity:** O(n): each index enters and leaves the deque once.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Brute force O(n·k) or a heap O(n log n) are fine to mention first. The **monotonic deque** is the stack trick from A3 with removal from both ends.
</details>

## B3 · Kth Largest Element in a Stream
`LC 703 · Easy`

**Why it might come up:** "Track the 3rd largest trade size as trades stream in." A size-k heap.

**Problem.** `KthLargest(k, nums)`; `add(val)` adds a value and returns the current k-th largest.

**Example.** k=3, nums=[4,5,8,2]: add(3)→4, add(5)→5, add(10)→5, add(9)→8, add(4)→8

In [ ]:
class KthLargest:
    def __init__(self, k: int, nums: list[int]):
        pass
    def add(self, val: int) -> int:
        pass

# --- tests ---
check_ops("KthLargest", ["KthLargest","add","add","add","add","add"], [[3,[4,5,8,2]],[3],[5],[10],[9],[4]], [None,4,5,5,8,8])

<details><summary>💡 <b>Hint</b></summary>

Keep a **min-heap of size k**: the k largest values seen. Its top is the k-th largest.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class KthLargest:
    def __init__(self, k: int, nums: list[int]):
        self.k, self.h = k, []
        for x in nums:
            self.add(x)

    def add(self, val: int) -> int:
        heapq.heappush(self.h, val)
        if len(self.h) > self.k:
            heapq.heappop(self.h)           # drop the smallest
        return self.h[0]
```

**Complexity:** O(log k) per add, O(k) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

"Top k largest" → **min**-heap of size k (counter-intuitive but key). Python's `heapq` is a min-heap; negate values for a max-heap.
</details>

## B4 · Kth Largest Element in an Array
`LC 215 · Medium`

**Why it might come up:** The non-streaming version. Know two answers: heap, and quickselect.

**Problem.** Return the k-th largest element (not necessarily distinct).

**Example.** `[3,2,1,5,6,4], k=2` → `5` · `[3,2,3,1,2,4,5,5,6], k=4` → `4`

In [ ]:
def findKthLargest(nums: list[int], k: int) -> int:
    pass

# --- tests ---
check(findKthLargest, [(([3,2,1,5,6,4], 2), 5), (([3,2,3,1,2,4,5,5,6], 4), 4), (([1], 1), 1)])

<details><summary>💡 <b>Hint</b></summary>

Size-k min-heap as in B3 (O(n log k)). Or quickselect: average O(n).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def findKthLargest(nums: list[int], k: int) -> int:
    h = []
    for x in nums:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)
    return h[0]

# One-liner you can mention: heapq.nlargest(k, nums)[-1]
# Quickselect (average O(n)): partition around a random pivot, recurse into one side only.
```

**Complexity:** O(n log k) with a heap; quickselect O(n) average, O(n²) worst.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Give the simple heap answer, then mention quickselect and its worst case. Sorting (O(n log n)) is a fine starting point to say out loud.
</details>

---
# C · Small design problems (same type as the Squarepoint key-value store)

## C1 · Time Based Key-Value Store
`LC 981 · Medium`

**Why it might come up:** "What was AAPL's price at 10:31?" A hashmap of sorted lists + binary search. Closely related to the Squarepoint question.

**Problem.** `set(key, value, timestamp)` (timestamps for a key always increase). `get(key, timestamp)` returns the value with the largest timestamp ≤ `timestamp`, or `""`.

**Example.** set("foo","bar",1), get("foo",1)→"bar", get("foo",3)→"bar", set("foo","bar2",4), get("foo",4)→"bar2", get("foo",5)→"bar2", get("foo",0)→""

In [ ]:
class TimeMap:
    def __init__(self):
        pass
    def set(self, key: str, value: str, timestamp: int) -> None:
        pass
    def get(self, key: str, timestamp: int) -> str:
        pass

# --- tests ---
check_ops("TimeMap", ["TimeMap","set","get","get","set","get","get","get"], [[],["foo","bar",1],["foo",1],["foo",3],["foo","bar2",4],["foo",4],["foo",5],["foo",0]], [None,None,"bar","bar",None,"bar2","bar2",""])

<details><summary>💡 <b>Hint</b></summary>

`dict key → list of (timestamp, value)`, already sorted because timestamps increase. `get` = `bisect_right` on the timestamps, then step one left.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class TimeMap:
    def __init__(self):
        self.ts = defaultdict(list)          # key -> [timestamps]
        self.vals = defaultdict(list)        # key -> [values]

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.ts[key].append(timestamp)
        self.vals[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        i = bisect.bisect_right(self.ts[key], timestamp)   # first ts > timestamp
        return self.vals[key][i - 1] if i else ""
```

**Complexity:** set O(1), get O(log n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

"Latest value at or before time t" = **binary search for the right edge** (`bisect_right - 1`). Ask: can timestamps arrive out of order? (Then you'd need insort or a sorted container.)
</details>

## C2 · LRU Cache
`LC 146 · Medium`

**Why it might come up:** The most common design question for interns anywhere. Already in notebook 02 (C6); repeat it until it's automatic.

**Problem.** O(1) `get(key)` (−1 if missing) and `put(key, value)`, evicting the least recently used key when over capacity.

**Example.** See the test.

In [ ]:
class LRUCache:
    def __init__(self, capacity: int):
        pass
    def get(self, key: int) -> int:
        pass
    def put(self, key: int, value: int) -> None:
        pass

# --- tests ---
check_ops("LRUCache", ["LRUCache","put","put","get","put","get","put","get","get","get"], [[2],[1,1],[2,2],[1],[3,3],[2],[4,4],[1],[3],[4]], [None,None,None,1,None,-1,None,-1,3,4])

<details><summary>💡 <b>Hint</b></summary>

`OrderedDict`: `move_to_end(key)` on every access; `popitem(last=False)` evicts the oldest. Without it: dict + doubly linked list (notebook 02 C6).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class LRUCache:
    def __init__(self, capacity: int):
        self.cap, self.d = capacity, OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.d:
            return -1
        self.d.move_to_end(key)
        return self.d[key]

    def put(self, key: int, value: int) -> None:
        self.d[key] = value
        self.d.move_to_end(key)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)
```

**Complexity:** O(1) per operation.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Hash map (find fast) + ordered list (reorder fast). Expect "now without OrderedDict": see notebook 02 C6.
</details>

## C3 · Logger Rate Limiter
`LC 359 · Easy`

**Why it might come up:** Rate limiting is everywhere in trading systems (max orders per second). A short warm-up design.

**Problem.** `shouldPrintMessage(timestamp, message)`: return True if the same message hasn't been printed in the last 10 seconds (timestamps are non-decreasing).

**Example.** (1,'foo')→True, (2,'bar')→True, (3,'foo')→False, (8,'bar')→False, (10,'foo')→False (only 9 s since 1), (11,'foo')→True

In [ ]:
class Logger:
    def __init__(self):
        pass
    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        pass

# --- tests ---
check_ops("Logger", ["Logger","shouldPrintMessage","shouldPrintMessage","shouldPrintMessage","shouldPrintMessage","shouldPrintMessage","shouldPrintMessage"], [[],[1,"foo"],[2,"bar"],[3,"foo"],[8,"bar"],[10,"foo"],[11,"foo"]], [None,True,True,False,False,False,True])

<details><summary>💡 <b>Hint</b></summary>

`next_allowed = {message: timestamp + 10}`. Allowed iff `timestamp >= next_allowed.get(message, 0)`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class Logger:
    def __init__(self):
        self.next_ok = {}                    # message -> earliest timestamp it may print again

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if timestamp < self.next_ok.get(message, 0):
            return False
        self.next_ok[message] = timestamp + 10
        return True
```

**Complexity:** O(1) per call; memory grows with distinct messages (follow-up: clean up old keys).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Clarify the boundary: is a gap of exactly 10 allowed? Here (1→11) is allowed and (1→10) isn't. Asking that question is exactly the "clarifying questions" tip from your email.
</details>

## C4 · Design Hit Counter
`LC 362 · Medium`

**Why it might come up:** "How many orders did this client send in the last 5 minutes?" A sliding time window with a deque.

**Problem.** `hit(timestamp)` records a hit; `getHits(timestamp)` returns hits in the past 300 seconds, i.e. timestamps in `(timestamp − 300, timestamp]`. Timestamps are non-decreasing.

**Example.** hit(1), hit(2), hit(3), getHits(4)→3, hit(300), getHits(300)→4, getHits(301)→3

In [ ]:
class HitCounter:
    def __init__(self):
        pass
    def hit(self, timestamp: int) -> None:
        pass
    def getHits(self, timestamp: int) -> int:
        pass

# --- tests ---
check_ops("HitCounter", ["HitCounter","hit","hit","hit","getHits","hit","getHits","getHits"], [[],[1],[2],[3],[4],[300],[300],[301]], [None,None,None,None,3,None,4,3])

<details><summary>💡 <b>Hint</b></summary>

Deque of timestamps. In `getHits`, pop from the left while `q[0] <= timestamp - 300`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class HitCounter:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0] <= timestamp - 300:
            self.q.popleft()
        return len(self.q)
```

**Complexity:** Amortized O(1). Follow-up for huge hit rates: store `(timestamp, count)` pairs, or a fixed array of 300 buckets.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Time-window counting = deque of timestamps, evict from the front. The same structure enforces "max N orders per second".
</details>

## C5 · Simple order book (add + match)
`Custom · Medium`

**Why it might come up:** The likeliest custom question on an execution desk. This is a simpler version than notebook 04 (no cancel).

**Problem.** `add(oid, side, price, qty)` with side `"B"` (buy) or `"S"` (sell). An incoming order trades against the opposite side **while prices cross** (buy price ≥ best sell price, or sell price ≤ best buy price), best price first, and **first-come-first-served** at the same price. Trades happen at the **resting** order's price. Leftover quantity rests in the book. Return the trades as `(resting_oid, incoming_oid, price, qty)`. Also `best_bid()` / `best_ask()` → price or `None`.

**Example.** See the tests: buys at 100 and 99, then a sell at 99 for 18 sweeps them.

In [ ]:
class SimpleOrderBook:
    def __init__(self):
        pass
    def add(self, oid, side, price, qty):
        pass
    def best_bid(self):
        pass
    def best_ask(self):
        pass

def scenario():          # test helper: don't change
    ob, out = SimpleOrderBook(), []
    out.append(ob.add(1, "B", 100, 10)); out.append(ob.add(2, "B", 100, 5)); out.append(ob.add(3, "B", 99, 7))
    out.append(ob.add(4, "S", 102, 8)); out.append((ob.best_bid(), ob.best_ask()))
    out.append(ob.add(5, "S", 99, 18)); out.append((ob.best_bid(), ob.best_ask()))
    out.append(ob.add(6, "B", 105, 10)); out.append((ob.best_bid(), ob.best_ask()))
    return out

# --- tests ---
check(scenario, [((), [[], [], [], [], (100, 102), [(1,5,100,10), (2,5,100,5), (3,5,99,3)], (99, 102), [(4,6,102,8)], (105, None)])])

<details><summary>💡 <b>Hint</b></summary>

Two heaps. Buys: `(-price, seq, [qty, oid])` so the highest price pops first. Sells: `(price, seq, [qty, oid])`. `seq` is an increasing counter, so equal prices pop in arrival order (FIFO). Keep qty in a list so you can reduce it in place for partial fills.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class SimpleOrderBook:
    def __init__(self):
        self.bids, self.asks, self.seq = [], [], 0      # heaps

    def add(self, oid, side, price, qty):
        if side == "B":
            book, opp, crosses = self.bids, self.asks, (lambda p: p <= price)
        else:
            book, opp, crosses = self.asks, self.bids, (lambda p: p >= price)
        trades = []
        while qty and opp:
            key, _, rest = opp[0]
            rest_px = key if side == "B" else -key       # undo the negation for bids
            if not crosses(rest_px):
                break
            fill = min(qty, rest[0])
            trades.append((rest[1], oid, rest_px, fill))
            qty -= fill
            rest[0] -= fill
            if rest[0] == 0:
                heapq.heappop(opp)
        if qty:
            self.seq += 1
            heapq.heappush(book, (-price if side == "B" else price, self.seq, [qty, oid]))
        return trades

    def best_bid(self):
        return -self.bids[0][0] if self.bids else None

    def best_ask(self):
        return self.asks[0][0] if self.asks else None
```

**Complexity:** add: O(log n) per resting order touched or inserted. best: O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Say the design before coding: "two priority queues, buys by highest price, sells by lowest, ties by arrival time." Likely follow-up: **cancel**. Keep `oid → entry` in a dict, set its qty to 0, and skip zero-qty entries when they reach the top (lazy deletion, same as A5). The full version is notebook 04, 6a.
</details>

---
# D · Classic warm-ups

## D1 · Two Sum
`LC 1 · Easy`

**Why it might come up:** The most common opener. 3 minutes, no hesitation.

**Problem.** Return indices of the two numbers adding up to `target`.

**Example.** `[2,7,11,15], 9` → `[0,1]`

In [ ]:
def twoSum(nums: list[int], target: int) -> list[int]:
    pass

# --- tests ---
check(twoSum, [(([2,7,11,15], 9), [0,1]), (([3,2,4], 6), [1,2]), (([3,3], 6), [0,1])])

<details><summary>💡 <b>Hint</b></summary>

One pass with a dict `value → index`; check for `target - x` before inserting x.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def twoSum(nums: list[int], target: int) -> list[int]:
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return []
```

**Complexity:** O(n), O(n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Hash map for "have I seen the complement?". Mention the O(n²) brute force first.
</details>

## D2 · Merge Intervals
`LC 56 · Medium`

**Why it might come up:** "Merge overlapping trading sessions / outage windows." Sort, then sweep.

**Problem.** Merge all overlapping intervals.

**Example.** `[[1,3],[2,6],[8,10],[15,18]]` → `[[1,6],[8,10],[15,18]]`

In [ ]:
def merge(intervals: list[list[int]]) -> list[list[int]]:
    pass

# --- tests ---
check(merge, [(([[1,3],[2,6],[8,10],[15,18]],), [[1,6],[8,10],[15,18]]), (([[1,4],[4,5]],), [[1,5]]), (([[1,4],[2,3]],), [[1,4]])])

<details><summary>💡 <b>Hint</b></summary>

Sort by start. If the next start ≤ the last merged end, extend the end with `max`; otherwise start a new interval.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def merge(intervals: list[list[int]]) -> list[list[int]]:
    intervals.sort(key=lambda iv: iv[0])
    out = []
    for s, e in intervals:
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out
```

**Complexity:** O(n log n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Sort first, then a single pass. Don't forget `max()` for an interval fully inside another.
</details>

## D3 · Merge k sorted feeds
`LC 23 · Hard (Medium in practice)`

**Why it might come up:** "Several exchanges each send trades sorted by time. Produce one time-ordered stream." Same as Merge k Sorted Lists.

**Problem.** Each feed is a list of `(timestamp, symbol)` sorted by timestamp. Merge them into one list sorted by timestamp (ties: lower feed index first).

**Example.** `[[(1,'A'),(4,'A')], [(2,'B'),(3,'B')], []]` → `[(1,'A'),(2,'B'),(3,'B'),(4,'A')]`

In [ ]:
def merge_feeds(feeds: list[list[tuple]]) -> list[tuple]:
    pass

# --- tests ---
check(merge_feeds, [(([[(1,'A'),(4,'A')], [(2,'B'),(3,'B')], []],), [(1,'A'),(2,'B'),(3,'B'),(4,'A')]), (([[(5,'X')], [(5,'Y')]],), [(5,'X'),(5,'Y')]), (([],), [])])

<details><summary>💡 <b>Hint</b></summary>

Heap holding one item per feed: `(timestamp, feed_index, position)`. Pop the smallest, output it, push that feed's next item.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def merge_feeds(feeds: list[list[tuple]]) -> list[tuple]:
    heap = [(f[0][0], i, 0) for i, f in enumerate(feeds) if f]
    heapq.heapify(heap)
    out = []
    while heap:
        ts, i, j = heapq.heappop(heap)
        out.append(feeds[i][j])
        if j + 1 < len(feeds[i]):
            heapq.heappush(heap, (feeds[i][j + 1][0], i, j + 1))
    return out
```

**Complexity:** O(N log k) for N total events across k feeds; O(k) heap.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**k-way merge = heap with one entry per source.** Putting the feed index in the tuple breaks ties deterministically and avoids comparing payloads.
</details>

---
# Final check (Sunday night)
- [ ] I can write A1, B1, C1, C2 from memory in under 10 minutes each
- [ ] I can explain the monotonic stack (A3) in two sentences
- [ ] I can say why "top k largest" uses a **min**-heap
- [ ] I did C5 on paper, out loud, in 25 minutes
- [ ] For every problem I said brute force → better idea → complexity *before* coding